# 1. ML Approach
- **Problem:** Kenyan youth struggle to verify political claims in English, Swahili, and Sheng.
- **Solution:** RAG system using official legal PDFs.
- **Models:** `SemanticChunker` (embedding-based semantic splitting), `text-embedding-3-small`, ChromaDB (MMR `k=3`), and `gpt-4o-mini`.

# 2. Loading Datasets
- **Knowledge Base:** 8 Kenyan legal PDFs (758 pages).
- **Test Data:** `lusamaki/Fake_News_Detection_System_9K` from Hugging Face.

In [ ]:
import os
from dotenv import load_dotenv

# Load API key
print("Exists:", os.path.exists("openai.env"))
load_dotenv("openai.env", override=True)
print("Key loaded:", bool(os.getenv("OPENAI_API_KEY")))

Exists: True
Key loaded: True


In [ ]:
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [ ]:
import sys
!{sys.executable} -m pip install pypdf

In [ ]:
# Load PDFs
docs_path = "./RAG_CONTEXT_DOCUMENTS"
documents = []

for filename in os.listdir(docs_path):
    if filename.endswith(".pdf"):
        file_path = os.path.join(docs_path, filename)
        try:
            loader = PyPDFLoader(file_path)
            docs = loader.load()
            documents.extend(docs)
            print(f"Loaded: {filename} ({len(docs)} pages)")
        except Exception as e:
            print(f"Error loading {filename}: {e}")

print(f"Total pages: {len(documents)}")

Loaded: Election Offences Act.pdf (14 pages)
Loaded: IEBC Election Operation Plan.pdf (71 pages)
Loaded: IEBC Voter Education Training Manual and Handbook.pdf (229 pages)
Loaded: Kenya Kwanza UDA Manifesto 2022.pdf (66 pages)
Loaded: Leadership and Integrity Act.pdf (21 pages)
Loaded: Political Parties Act.pdf (41 pages)
Loaded: The Elections (General) Regulations.pdf (123 pages)
Loaded: The_Constitution_of_Kenya_2010.pdf (193 pages)
Total pages: 758


In [ ]:
# Load Hugging Face evaluation dataset
try:
    from datasets import load_dataset
    hf_dataset = load_dataset("lusamaki/Fake_News_Detection_System_9K", split="train")
    bench_df = pd.DataFrame(hf_dataset)
    print(f"Loaded {len(bench_df)} claims.")
    print(bench_df['label'].value_counts())
except Exception as e:
    print("Using offline sample:")
    bench_df = pd.DataFrame({
        "text": [
            "A voter can legally register in two constituencies as long as they only vote once on election day.",
            "Kulingana na Katiba ya Kenya Kifungu cha 26, kila mtu ana haki ya kuishi tangu kutungwa mimba.",
            "Wasee ukipatikana na ballot paper imemarkwa nje ya polling station ni kesi chini ya Election Offences Act!",
            "UDA manifesto states that political patrons and tribal kingpins are required to lead national parties.",
            "Employers are legally required under the Election Offences Act to give employees reasonable time to vote."
        ],
        "label": ["fake", "fact", "fact", "fake", "fact"]
    })

# Convert to True/False
binary_bench_df = bench_df[bench_df['label'].isin(['fact', 'fake'])].copy()
binary_bench_df['verified_target'] = binary_bench_df['label'].map({'fact': True, 'fake': False})
display(binary_bench_df.head(5))

Loaded 6300 claims.
label
fact       2100
neutral    2100
fake       2100
Name: count, dtype: int64


,Unnamed: 0,tweet,label,text,language,contains_emoji,verified_target
0,5266,@user @user So he only worked from June 1st to...,fact,worked june st alter presidential elections re...,sheng,True,True
1,4574,It s 61 days to 2022 general elections to end ...,fact,s days general elections end martharao sababu ...,sheng,False,True
3,1967,There will be no need of panic\nAvoid incompet...,fact,need panicavoid incompetence vote wiselykenyad...,sheng,False,True
4,118,IEBC Kenya imeweka mbinu mbili kuu za kuwatamb...,fact,iebc kenya imeweka mbinu mbili kuu za kuwatamb...,Swahili,False,True
5,1521,@user Show us your tallying centre Bladfwakin ...,fact,tallying centre bladfwakin wewe,sheng,False,True


# 4. Data Cleaning

In [ ]:
# Clean PDF text
def clean_legal_document_text(text: str) -> str:
    text = re.sub(r'www\.kenyalaw\.org.*?\n', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'FRBR URI:.*?\n', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'By Kenya Law and Laws\.Africa.*?\n', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\.{4,}\s*\d*', ' ', text)
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    text = text.replace('\xa0', ' ').replace('ﬁ', 'fi').replace('ﬂ', 'fl')
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    return text.strip()

# Clean user queries
def clean_social_media_query(query: str) -> str:
    query = re.sub(r'http\S+|www\.\S+', '', query)
    query = re.sub(r'@[A-Za-z0-9_]+', '', query)
    query = re.sub(r'\s+', ' ', query).strip()
    return query

cleaned_documents = []
for doc in documents:
    cleaned_text = clean_legal_document_text(doc.page_content)
    if len(cleaned_text) > 100 and "Creative Commons Attribution NonCommercial" not in cleaned_text:
        cleaned_documents.append(Document(page_content=cleaned_text, metadata=doc.metadata))

print(f"Cleaned pages: {len(cleaned_documents)}")

# Split into 500-token chunks
modelling_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)
candidate_chunks = modelling_splitter.split_documents(cleaned_documents)

# Remove duplicates and very short chunks
seen_contents = set()
final_modelling_chunks = []
for chunk in candidate_chunks:
    norm_text = chunk.page_content.strip()
    if len(norm_text) >= 80 and norm_text not in seen_contents:
        seen_contents.add(norm_text)
        final_modelling_chunks.append(chunk)

print(f"Final chunks: {len(final_modelling_chunks)}")

Cleaned pages: 727
Final chunks: 881


# 3. EDA and Chunking

In [ ]:
# a) Fixed Splitting
fixed_splitter = CharacterTextSplitter(separator="", chunk_size=1000, chunk_overlap=100)
fixed_chunks = fixed_splitter.split_documents(documents)
chunk_lengths = [len(chunk.page_content) for chunk in fixed_chunks]
print(f"Fixed chunks: {len(chunk_lengths)}")

Fixed chunks: 1652


In [ ]:
# b) Recursive Splitting
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=500,
    chunk_overlap=50
)
recursive_chunks = recursive_splitter.split_documents(documents)
chunksizes = [len(rc.page_content) for rc in recursive_chunks]
print(f"Recursive chunks: {len(chunksizes)}")

Recursive chunks: 3012


In [ ]:
# c) Semantic Splitting on Cleaned Documents
semantic_splitter = SemanticChunker(OpenAIEmbeddings())
semantic_chunks = semantic_splitter.split_documents(cleaned_documents)
semantic_chunk_lengths = [len(chunk.page_content) for chunk in semantic_chunks]
print(f"Semantic chunks: {len(semantic_chunks)}")

**Observations:**
- PDFs have table of contents, headers, and links that add noise.
- Fixed splitting cuts words in half, while recursive keeps sentences together.
- Semantic chunks are very uneven in size.
- Some words are broken by hyphens and some chunks repeat.

# 5. Base Model and Testing

In [ ]:
# Store semantic chunks in ChromaDB vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
my_persist_directory = "db/chroma_db"
vector_store = Chroma.from_documents(
    documents=semantic_chunks,
    embedding=embeddings,
    persist_directory=my_persist_directory,
)
print("Vector Store Created with Semantic Chunks")

In [ ]:
# Test MMR retrieval
question = "What does the UDA manifesto say?"
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 10, "lambda_mult": 0.7}
)
matches = mmr_retriever.invoke(question)
for idx, match in enumerate(matches, 1):
    print(f"Match {idx}: {match.page_content[:180]}...\n")

In [ ]:
# Generate answer with LLM
context = "\n\n".join(match.page_content for match in matches)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
prompt = f"""
Answer the question using only the context below.
If the context does not contain the answer, say so:

Context:
{context}

Question: {question}
"""
answer = model.invoke(prompt)
print(answer.content)

### Chat History Function

In [ ]:
chat_history = []

def ask_question(user_question):
    cleaned_query = clean_social_media_query(user_question)
    if chat_history:
        messages = [
            SystemMessage(content="Given chat history, rewrite the new question to be standalone. Return only the question.")
        ] + chat_history + [HumanMessage(content=f"New Question: {cleaned_query}")]
        search_question = model.invoke(messages).content.strip()
    else:
        search_question = cleaned_query

    retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 10, "lambda_mult": 0.7})
    docs = retriever.invoke(search_question)
    context = "\n".join(f"- {doc.page_content}" for doc in docs)

    combined_input = f"""
    Based on these documents, answer: {user_question}
    Documents:
    {context}
    If the answer is not in the documents, say so.
    """
    messages = [SystemMessage(content="You are a helpful assistant."), HumanMessage(content=combined_input)]
    answer = model.invoke(messages).content

    chat_history.append(HumanMessage(content=user_question))
    chat_history.append(AIMessage(content=answer))
    return answer

### Model Validation

In [ ]:
# 1. K-Nearest Neighbors (KNN) Classifier Evaluation on Embeddings
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# Convert semantic chunks and benchmark samples into vector features for KNN validation
train_embeddings = embeddings.embed_documents([doc.page_content for doc in semantic_chunks[:300]])
train_labels = [i % 2 for i in range(len(train_embeddings))] # Binary classification proxy for demo training

test_embeddings = embeddings.embed_documents(binary_bench_df['text'].head(50).tolist())
test_labels = binary_bench_df['verified_target'].head(50).astype(int).tolist()

# Train K-Nearest Neighbors classifier
knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
knn.fit(train_embeddings, train_labels)

pred_labels = knn.predict(test_embeddings)
knn_acc = accuracy_score(test_labels, pred_labels)
print(f"KNN Classifier Accuracy (k=5, cosine metric): {knn_acc*100:.1f}%")


In [ ]:
# 3. Test out-of-domain question
ood_query = "What was the exact rainfall in Kisumu County in November 2025?"
print("Query:", ood_query)
print("Response:", ask_question(ood_query))

In [ ]:
# 4. Test chat history
demo_questions = [
    "What does the constitution state on Murder",
    "What is it",
    "What does the IEBC voter regulation document say on things that may be election irregularities"
]
chat_history.clear()
for q in demo_questions:
    print(f"\nQuestion: {q}")
    print(ask_question(q))

# 6. Model Comparison and Validation Strategy
To rigorously validate our Semantic RAG architecture, we compare four distinct model configurations on the same evaluation benchmark:
1. **Zero-Shot LLM:** `gpt-4o-mini` without RAG context.
2. **BM25 RAG:** Traditional keyword matching instead of vector search.
3. **Recursive RAG:** RAG pipeline using recursive character splitting boundaries.
4. **Proposed Semantic RAG Base:** Cleaned semantic chunks + ChromaDB + MMR (`k=3`) + `gpt-4o-mini`.